# HARD_BENCH — a frozen-small-LLM FAILURE benchmark (inference-only)

**Goal:** 20 categories × 10 = **200** examples with **canonical, machine-verifiable** answers, calibrated so a **frozen Qwen2.5-1.5B-Instruct** scores **~0% correct**. Categories are reverse-engineered from documented small-LLM failure modes: letter/word counting, multi-digit multiplication, base conversion, modular exponentiation, sorting/kth-element, string reversal, ROT13/Caesar ciphers, nested-parenthesis depth, date arithmetic, nth-term sequences, needle-in-a-long-list, acronym-from-words.

**Why it should read ~0%:** every answer is computed by a deterministic Python solver (no LLM judge), grading is **strict** (a numeric answer must be the *single* committed number equal to the canonical; a string must *equal* it — no list/substring/echo gaming), and every answer sits **above the blind-guess ceiling** so a non-reasoning guess never hits. The hard part needs a loop + exact arithmetic a 1.5B can't do in one forward pass.

## 0 · GPU check

In [ ]:
!nvidia-smi || echo 'NO GPU — runs on CPU (slow). Colab: Runtime -> Change runtime type -> T4 GPU, then re-run.'

## 1 · Clone & install (pulls the latest commit)

In [ ]:
import os
!rm -rf small_fable
!git clone -q https://github.com/sinha-k-prat/small_fable.git
%cd small_fable
!git log --oneline -1
!pip install -q -r requirements.txt
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('\nReady.')

## 2 · Build the benchmark (deterministic, CPU, instant)

In [ ]:
!python -u tools/gen_hard_bench.py --seed 20240621 --out hard_bench.jsonl
import json, collections
rows=[json.loads(l) for l in open('hard_bench.jsonl')]
print('rows:', len(rows), '| categories:', len(set(r['category'] for r in rows)))
for r in rows[:3]:
    print('\n['+r['category']+']', r['question'][:130].replace(chr(10),' '), '-> answer', r['answer'])

## 2.5 · Calibration self-test (no model) — must PASS before the run
Confirms: canonical answers score **100%** under the strict grader, and **no blind guess** scores above ~0% on any category (so a non-reasoning model truly can't score). If this FAILS, the benchmark isn't calibrated.

In [ ]:
!python tools/hard_bench_selftest.py hard_bench.jsonl

## 3 · Run the benchmark on the frozen model
A100 -> `bf16`; T4 -> `fp16`. Expectation: **overall ~0%**.

In [ ]:
DTYPE='fp16'  # 'bf16' on A100
!python -u hard_bench_run.py --data hard_bench.jsonl --base Qwen/Qwen2.5-1.5B-Instruct \
    --dtype $DTYPE --device cuda --max_new_tokens 256 --out hard_bench_out

## 4 · Per-category accuracy (expect ~0% everywhere)

In [ ]:
from IPython.display import Image, display
import os, json
png='hard_bench_out/hard_bench.png'
display(Image(filename=png)) if os.path.exists(png) else print('[!] plot not found:', png)
res=json.load(open('hard_bench_out/results.json'))
print('OVERALL accuracy:', f"{res.get('overall_accuracy','?')}")
print('marker_rate:', res.get('marker_rate'))